<a href="https://colab.research.google.com/github/Obulreddy0/pyspark-practice/blob/master/Day2_practise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyspark -q

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Day1_Practice") \
    .master("local[*]") \
    .getOrCreate()

print(f"✅ Spark {spark.version} ready!")

✅ Spark 4.0.4 ready!


In [2]:
!git clone https://github.com/Obulreddy0/pyspark-practice.git

customers_df = spark.read.option("header", "true").option("inferSchema", "true") \
    .csv("pyspark-practice/data/customers.csv")

customers_df.show()

Cloning into 'pyspark-practice'...
remote: Enumerating objects: 39, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 39 (delta 12), reused 20 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (39/39), 14.78 KiB | 841.00 KiB/s, done.
Resolving deltas: 100% (12/12), done.
+-----------+-------------+-----------------+---------+-----------+----+-----------+---------+
|customer_id|         name|            email|     city|      state| age|signup_date|is_active|
+-----------+-------------+-----------------+---------+-----------+----+-----------+---------+
|       C001| Rahul Sharma|  rahul@email.com|   Mumbai|Maharashtra|  28| 2023-01-15|     true|
|       C002|  Priya Patel|  priya@email.com|Ahmedabad|    Gujarat|  34| 2023-02-20|     true|
|       C003|   Amit Kumar|   amit@email.com|    Delhi|      Delhi|  45| 2022-11-10|    false|
|       C004|  Sneha Reddy|  sneha@email.com|Hyderabad|  Telangana|  29| 2023-03-05| 

In [3]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("pyspark-practice/data/orders.csv")

df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- status: string (nullable = true)



In [4]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("discount", DoubleType(), True),
    StructField("status", StringType(), True),
])

df = spark.read \
    .option("header", "true") \
    .schema(orders_schema) \
    .csv("pyspark-practice/data/orders.csv")

df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- status: string (nullable = true)



In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

spark = SparkSession.builder.appName("ETL_Example").master("local[*]").getOrCreate()

# 1. Define schema
schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("discount", DoubleType(), True),
    StructField("status", StringType(), True),
])

# 2. Read CSV with explicit schema
orders_df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv("pyspark-practice/data/orders.csv")

# 3. Quick check
print(f"Rows: {orders_df.count()}, Columns: {len(orders_df.columns)}")
orders_df.show(5)

# 4. Write as Parquet (production format)
orders_df.write.mode("overwrite").parquet("output/orders_parquet")

print("✅ CSV → Parquet conversion done!")


Rows: 25, Columns: 8
+--------+-----------+----------+----------+--------+----------+--------+---------+
|order_id|customer_id|product_id|order_date|quantity|unit_price|discount|   status|
+--------+-----------+----------+----------+--------+----------+--------+---------+
|   O1001|       C001|      P101|2023-06-01|       2|    1500.0|     0.1|delivered|
|   O1002|       C002|      P102|2023-06-02|       1|    2500.0|     0.0|delivered|
|   O1003|       C001|      P103|2023-06-03|       3|     800.0|    0.05|  shipped|
|   O1004|       C003|      P101|2023-06-05|       1|    1500.0|     0.0|cancelled|
|   O1005|       C004|      P104|2023-06-07|       2|    3200.0|    0.15|delivered|
+--------+-----------+----------+----------+--------+----------+--------+---------+
only showing top 5 rows
✅ CSV → Parquet conversion done!


In [10]:
# Read products.csv using inferSchema. Print the schema. Now read it again with a manual schema using StructType. Compare both schemas — are the types the same?

df = spark.read.option("header", "true").option("inferSchema", "true") \
    .csv("pyspark-practice/data/products.csv")

df.printSchema()

schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("stock_qty", IntegerType(), True),
])

df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv("pyspark-practice/data/products.csv")

df.printSchema()


root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- stock_qty: integer (nullable = true)

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- stock_qty: integer (nullable = true)

